# Entraînement YOLO-OBB (Module 3 — Orientation des composants)

Ce notebook prépare l'environnement, vérifie la structure de votre dataset OBB (`data.yaml` et labels au format OBB), entraîne un modèle **YOLO-OBB** (`yolo11n-obb.pt` ou `yolov8n-obb.pt`), évalue les performances et sauvegarde le modèle final (`best.pt`).

## Bloc 1 — Installation / Imports & Configuration des chemins

In [9]:
# Décommentez si ultralytics n'est pas installé dans votre environnement :
# !pip install -U ultralytics pyyaml

from pathlib import Path
import yaml
import torch
from ultralytics import YOLO

# Chemin vers votre fichier data.yaml du dataset OBB
DATA_YAML = Path(r"D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml").resolve()

# Modèle pré-entraîné OBB de base ('yolo11n-obb.pt', 'yolo11s-obb.pt', 'yolov8n-obb.pt', etc.)
BASE_MODEL = "yolo11n-obb.pt"

# Hyperparamètres d'entraînement



DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Device utilisé : {DEVICE}")
print(f"data.yaml cible : {DATA_YAML}")

Device utilisé : cpu
data.yaml cible : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml


## Bloc 2 — Vérification du dataset OBB (`data.yaml` et fichiers `.txt`)

In [10]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f"Fichier data.yaml introuvable : {DATA_YAML}")

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("Contenu de data.yaml :")
print(yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True))

dataset_root = Path(data_cfg.get("path", DATA_YAML.parent))
if not dataset_root.is_absolute():
    dataset_root = (DATA_YAML.parent / dataset_root).resolve()

for split in ["train", "val", "test"]:
    split_entry = data_cfg.get(split)
    if not split_entry:
        continue
    img_dir = Path(split_entry)
    if not img_dir.is_absolute():
        img_dir = (dataset_root / img_dir).resolve()
    lbl_dir = Path(str(img_dir).replace("images", "labels"))
    n_imgs = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"Split '{split:5s}' -> images: {img_dir} ({n_imgs} fichiers) | labels: {lbl_dir} ({n_lbls} .txt)")

    # Aperçu d'un fichier label pour vérifier le format OBB (classe + 8 coordonnées normalisées)
    if lbl_dir.exists():
        sample_lbls = sorted(lbl_dir.glob("*.txt"))
        if sample_lbls:
            with open(sample_lbls[0], "r", encoding="utf-8") as lf:
                lines = [ln.strip() for ln in lf.readlines() if ln.strip()]
            print(f"  Exemple label ({sample_lbls[0].name}, {len(lines)} ligne(s)) : {lines[:2]}")

Contenu de data.yaml :
train: train/images
val: valid/images
test: test/images
names:
  0: cable
  1: cable-tie
  2: clip
  3: connector
  4: label

Split 'train' -> images: D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\train\images (234 fichiers) | labels: D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\train\labels (234 .txt)
  Exemple label (100_jpg.rf.398cc1c3123dd3256d16e34336357140.txt, 7 ligne(s)) : ['3 0.6629792692912119 0.2905584980444177 0.6470786856247905 0.12369592790872193 0.39426446619353894 0.14778697268202595 0.4101650498599604 0.31464954281772173', '3 0.8279479253177744 0.5129185618397727 0.801919772667554 0.3919389968072207 0.5676555047437836 0.4423397907294951 0.593683657394004 0.5633193557620471']
Split 'val  ' -> images: D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\valid\images (22 fichiers) | labels: D:\S5\Projet_d_exper

## Bloc 3 — Entraînement du modèle YOLO-OBB

In [11]:
model = YOLO(BASE_MODEL)

In [12]:
PROJECT_DIR = Path(r"D:\S5\Projet_d_expertise\projet\models").resolve()
RUN_NAME = "03_orientation_des_composants" 

In [13]:
train_results = model.train(
    data=str(DATA_YAML),
    epochs=10,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

In [14]:
best_model_path = Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
print(f"Entraînement terminé. Meilleur modèle : {best_model_path}")

Entraînement terminé. Meilleur modèle : D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants\weights\best.pt
